In [1]:
import os
import sys
from pathlib import Path

# Automatically locate and add the 'src' folder to sys.path
project_root = Path(os.path.abspath(''))
src_path = project_root / "src"
if src_path not in sys.path:
    sys.path.insert(0, str(src_path))

In [2]:
import numpy as np
import SimpleITK as sitk
import vtk
from vtk.util import numpy_support

from typing import Union, Optional, Tuple
import warnings

In [34]:
stl_path = Path(f'/run/user/1003/gvfs/smb-share:server=tierra.cnic.es,share=sc/LAB_FSC/LAB/PROJECTS/PESA/subproyectos/PESA_Fat/Data/STL/PESA_PESA1225/Sin descripción del estudio_01-18-2013/CT Viewer resultados 03-ene.-2025/Tissue 1 PESA PESA1225.stl')
ref_nifti_path = Path(f'/run/user/1003/gvfs/smb-share:server=tierra.cnic.es,share=sc/LAB_FSC/LAB/PROJECTS/PESA/subproyectos/PESA_Fat/Data/Original_Data_nifti_IMS/V1/S41150/BATCH_XC_FINAL_MR_803.nii')
# ref_nifti_path = Path(f'/run/user/1003/gvfs/smb-share:server=tierra.cnic.es,share=sc/LAB_FSC/LAB/PROJECTS/PESA/subproyectos/PESA_Fat/Data/Original_Data_nifti_IMS/V1_test/BATCH_XC_FINAL_MR_803_2.nii')
output_path = Path(f'/run/user/1003/gvfs/smb-share:server=tierra.cnic.es,share=sc/LAB_FSC/LAB/PROJECTS/PESA/subproyectos/PESA_Fat/Data/Original_Data_nifti_IMS/STL/PESA_PESA1225/Tissue_1_PESA1225.nii')

In [27]:
import logging
import functools
import numpy as np
import vtk

# Decorator to update VTK filters

def update(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        vtk_filter = func(*args, **kwargs)
        vtk_filter.Update()
        return vtk_filter
    return wrapper


@update
def nb_read_stl(filename):
    reader = vtk.vtkSTLReader()
    reader.SetFileName(str(filename))
    return reader


@update
def nb_read_nifti(filename):
    reader = vtk.vtkNIFTIImageReader()
    reader.SetFileName(str(filename))
    return reader


@update
def nb_write_nifti(filename, image, transform_matrix, qfac):
    writer = vtk.vtkNIFTIImageWriter()
    writer.SetFileName(str(filename))
    writer.SetInputData(image)
    # Use SForm if QForm is None
    if transform_matrix is not None:
        writer.SetSFormMatrix(transform_matrix)
    writer.SetQFac(qfac)
    return writer


def nb_get_surface_origin(bounds, spacing):
    return tuple(bounds[2*i] + (s / 2.0) for i, s in enumerate(spacing))


def nb_get_surface_dimensions(bounds, spacing):
    dims = [int((bounds[2*i+1] - bounds[2*i]) // spacing[i]) for i in range(3)]
    return tuple(dims)


def nb_get_origin_from_qform_matrix(QFormMatrix):
    offset = [QFormMatrix.GetElement(i, 3) for i in range(3)]
    sign = [QFormMatrix.GetElement(i, i) for i in range(3)]
    return tuple(s * o for s, o in zip(sign, offset))

def nb_get_origin_from_matrix(matrix):
    """Extract origin from QForm or SForm matrix"""
    if matrix is None:
        return (0.0, 0.0, 0.0)
    offset = [matrix.GetElement(i, 3) for i in range(3)]
    sign = [matrix.GetElement(i, i) for i in range(3)]
    return tuple(s * o for s, o in zip(sign, offset))

def nb_get_reference_information_from_image(reader, surface_bounds):
    spacing = reader.GetOutput().GetSpacing()
    surface_dimensions = nb_get_surface_dimensions(surface_bounds, spacing)
    surface_origin = nb_get_surface_origin(surface_bounds, spacing)
    
    # Try QForm first, fallback to SForm
    qform = reader.GetQFormMatrix()
    sform = reader.GetSFormMatrix()
    transform_matrix = qform if qform is not None else sform
    reference_origin = nb_get_origin_from_matrix(transform_matrix)

    config = {
        'Reference Dimensions': reader.GetOutput().GetDimensions(),
        'Surface Dimensions': surface_dimensions,
        'Spacing': spacing,
        'Reference Origin': reference_origin,
        'Surface Origin': surface_origin,
        'Direction': reader.GetOutput().GetDirectionMatrix(),
        'QFormMatrix': qform,
        'SFormMatrix': sform,
        'TransformMatrix': transform_matrix,  # The one we're actually using
        'QFac': reader.GetQFac(),
    }
    return config


def nb_init_vtk_image(spacing, dimensions, origin, direction, constant_value=1):
    vtkImage = vtk.vtkImageData()
    vtkImage.SetSpacing(spacing)
    vtkImage.SetDimensions(dimensions)
    vtkImage.SetDirectionMatrix(direction)
    vtkImage.SetOrigin(origin)
    vtkImage.AllocateScalars(vtk.VTK_UNSIGNED_CHAR, 1)

    scalars = vtkImage.GetPointData().GetScalars()
    try:
        scalars.Fill(constant_value)
    except AttributeError:
        for i in range(scalars.GetNumberOfTuples()):
            scalars.SetTuple1(i, constant_value)
    return vtkImage


@update
def nb_vtk_polydata2imagestencil(polydata, origin, spacing, extent):
    poly2stenc = vtk.vtkPolyDataToImageStencil()
    poly2stenc.SetInputData(polydata)
    poly2stenc.SetOutputOrigin(origin)
    poly2stenc.SetOutputSpacing(spacing)
    poly2stenc.SetOutputWholeExtent(extent)
    return poly2stenc


@update
def nb_get_image_stencil(vtk_image, poly2stencil=None, bkg=0):
    image_stencil = vtk.vtkImageStencil()
    image_stencil.SetInputData(vtk_image)
    if poly2stencil is not None:
        image_stencil.SetStencilConnection(poly2stencil.GetOutputPort())
    image_stencil.ReverseStencilOff()
    image_stencil.SetBackgroundValue(bkg)
    return image_stencil


@update
def nb_translate_image(image, offset=(0., 0., 0.), bkg_level=0):
    transform = vtk.vtkTransform()
    transform.Translate(*offset)

    reslice = vtk.vtkImageReslice()
    reslice.SetResliceTransform(transform)
    reslice.SetInterpolationModeToNearestNeighbor()
    reslice.SetInputData(image)
    reslice.SetOutputSpacing(image.GetSpacing())
    reslice.SetOutputOrigin(image.GetOrigin())
    reslice.SetOutputExtent(image.GetExtent())
    reslice.SetBackgroundLevel(bkg_level)
    return reslice


@update
def nb_change_image_information(image, origin):
    changer = vtk.vtkImageChangeInformation()
    changer.SetInputData(image)
    changer.SetOutputOrigin(origin)
    return changer


In [35]:
from imaging import imread


img, meta = imread(ref_nifti_path)
print(f"Image shape: {img.shape}")
print(f"Image resolution: {meta['x_res']} {meta['y_res']} {meta['z_res']}")
print(f"Image affine:\n {meta['affine']}")

Image shape: (320, 320, 21)
Image resolution: 1.875 1.875 6.0
Image affine:
 [[ -1.875       -0.           0.         299.0625    ]
 [ -0.          -1.875        0.         299.0625    ]
 [  0.           0.          -6.         182.76231384]
 [  0.           0.           0.           1.        ]]


In [29]:
# Run the notebook-adapted pipeline with fixed functions
surface = nb_read_stl(stl_path)
reference = nb_read_nifti(ref_nifti_path)

surface_bounds = surface.GetOutput().GetBounds()
config = nb_get_reference_information_from_image(reference, surface_bounds)

print("Using transform matrix:", "QForm" if config['QFormMatrix'] is not None else "SForm")
print("Reference origin:", config['Reference Origin'])
print("Surface origin:", config['Surface Origin'])

vtk_image = nb_init_vtk_image(
    spacing=config['Spacing'],
    dimensions=config['Reference Dimensions'],
    origin=config['Reference Origin'],
    direction=config['Direction'],
    constant_value=1,
)

poly2stencil = nb_vtk_polydata2imagestencil(
    polydata=surface.GetOutput(),
    origin=config['Surface Origin'],
    spacing=config['Spacing'],
    extent=vtk_image.GetExtent(),
)

image_stencil = nb_get_image_stencil(vtk_image=vtk_image, poly2stencil=poly2stencil, bkg=0)

offset = np.asarray(config['Reference Origin']) - np.asarray(config['Surface Origin'])
print("Translation offset:", offset)
translated = nb_translate_image(image_stencil.GetOutput(), tuple(offset))
translated = nb_change_image_information(translated.GetOutput(), (0., 0., 0.))

nb_write_nifti(
    filename=str(output_path),
    image=translated.GetOutput(),
    transform_matrix=config['TransformMatrix'],
    qfac=config['QFac']
)

print('Wrote:', output_path)

# Quick validation
arr = sitk.GetArrayFromImage(sitk.ReadImage(str(output_path)))
print(f'Output shape: {arr.shape}, nonzero voxels: {int((arr > 0).sum())}, dtype: {arr.dtype}')


Using transform matrix: SForm
Reference origin: (-299.0625, -299.0625, 62.76231384277344)
Surface origin: (-168.6754913330078, -107.55633544921875, 67.74137878417969)
Translation offset: [-130.38700867 -191.50616455   -4.97906494]
Wrote: /run/user/1003/gvfs/smb-share:server=tierra.cnic.es,share=sc/LAB_FSC/LAB/PROJECTS/PESA/subproyectos/PESA_Fat/Data/Original_Data_nifti_IMS/STL/PESA_PESA1225/Tissue_1_PESA1225.nii
Output shape: (21, 320, 320), nonzero voxels: 96474, dtype: uint8
